In [9]:
import pandas as pd
import nltk
import torch
from transformers import BertTokenizer, BertForSequenceClassification
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from nltk.tokenize import sent_tokenize
import json
import re
from huggingface_hub import login


In [11]:
tokenizer = AutoTokenizer.from_pretrained("gtfintechlab/FOMC-RoBERTa")
model = AutoModelForSequenceClassification.from_pretrained("gtfintechlab/FOMC-RoBERTa")

model.eval()

pytorch_model.bin:   1%|          | 10.5M/1.42G [00:00<?, ?B/s]

RobertaForSequenceClassification(
  (roberta): RobertaModel(
    (embeddings): RobertaEmbeddings(
      (word_embeddings): Embedding(50265, 1024, padding_idx=1)
      (position_embeddings): Embedding(514, 1024, padding_idx=1)
      (token_type_embeddings): Embedding(1, 1024)
      (LayerNorm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): RobertaEncoder(
      (layer): ModuleList(
        (0-23): 24 x RobertaLayer(
          (attention): RobertaAttention(
            (self): RobertaSdpaSelfAttention(
              (query): Linear(in_features=1024, out_features=1024, bias=True)
              (key): Linear(in_features=1024, out_features=1024, bias=True)
              (value): Linear(in_features=1024, out_features=1024, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): RobertaSelfOutput(
              (dense): Linear(in_features=1024, out_features=1024, bias=Tru

In [12]:
# Define a word list concerning tariff to focus the sentiment only on segments about tariff
tariff_keywords = [
    # Direct mentions
    'tariff', 'tariffs', 'import duty', 'import duties', 'customs duty', 'customs duties', 'export tariffs', 'retaliatory tariffs',
    
    # Trade barriers more broadly
    'trade barrier', 'trade barriers', 'protectionism', 'trade protectionism', 'trade restrictions', 'import restrictions',
    
    # Trade policy/negotiation terms
    'trade policy', 'trade negotiation', 'trade talks', 'trade deal', 'trade agreement', 'trade dispute', 'trade tensions', 'trade war', 'trade wars',
    
    # Retaliation-related
    'retaliation', 'retaliatory measures', 'retaliatory action', 'countertariffs', 'counter tariffs', 
    
    # Specific regions (contextually strong)
    'us-china trade', 'china tariffs', 'us tariffs', 'eu tariffs', 'mexico tariffs', 'canada tariffs',

    # Indirect but common Fed speech phrases
    'escalation in trade tensions', 'uncertainty around trade policy', 'trade uncertainty', 'global trade tensions', 
    'impact of tariffs', 'effects of tariffs', 'tariff-related uncertainty', 'supply chain disruption', 'supply chain risk',
    
    # Broader related
    'import taxes', 'export taxes', 'section 301', 'section 232',  # actual U.S. trade law actions
    'world trade organization', 'wto ruling', 'trade imbalances', 'bilateral trade deficit', 'bilateral trade surpluses'
]

In [ ]:
# Use FinBert to calculate the sentiment score
def predict_sentiment(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=512)
    outputs = model(**inputs)
    logits = outputs.logits
    # probs = torch.nn.functional.softmax(outputs.logits, dim=-1)
    # probs = probs.detach().numpy()[0]  # [positive, neutral, negative]
    probabilities = torch.softmax(logits, dim=1).detach().cpu().numpy()[0]
    
    hawkish_score = probabilities[1]  # if label 1 = hawkish
    dovish_score = probabilities[0] 
    
    return {
        'positive': hawkish_score,
        'neutral': probabilities[0],
        'negative': dovish_score
    }
    '''
    return {
        'positive': probs[0],
        'neutral': probs[1],
        'negative': probs[2]
    }'''

In [14]:
# Check if the segment is related to tariff
def is_tariff_related(text):
    text = text.lower()
    text = re.sub(r'[^a-z\s]', ' ', text)  # remove punctuation, numbers, keep spaces
    text = re.sub(r'\s+', ' ', text)  # collapse multiple spaces
    return any(keyword in text for keyword in tariff_keywords)

In [15]:
def analyze_document(date, filename = "", full_text = ""):
    # To split the full text into segments first
    segments = sent_tokenize(full_text)
    results = []
    
    # Analyze each segment and save the result
    for segment in segments:
        sentiment = predict_sentiment(segment)
        is_tariff = is_tariff_related(segment)
        
        results.append({
            'segment': segment,
            'positive': sentiment['positive'],
            'neutral': sentiment['neutral'],
            'negative': sentiment['negative'],
            'is_tariff': is_tariff
        })
        
    df = pd.DataFrame(results)

    # Calculate the result of the full document
    total_segments = len(df)
    hawkish = (df['positive'] > df['negative']).sum()
    dovish = (df['negative'] > df['positive']).sum()
    neutral = df['neutral'].sum()

    hawkish_pct = hawkish / total_segments * 100
    dovish_pct = dovish / total_segments * 100
    neutral_pct = (neutral / total_segments) * 100

    avg_polarity_score = (df['positive'] - df['negative']).mean()

    # Tariff Related Statistics
    tariff_df = df[df['is_tariff']]
    tariff_mentions = len(tariff_df)
    tariff_pct_of_doc = tariff_mentions / total_segments * 100 if total_segments > 0 else 0
    tariff_polarity_score = (tariff_df['positive'] - tariff_df['negative']).mean() if not tariff_df.empty else 0

    summary = {
        'Meeting Date': date,
        'Filename': filename,
        'Total Segments': total_segments,
        'Hawkish %': hawkish_pct,
        'Dovish %': dovish_pct,
        'Neutral %': neutral_pct,
        'Polarity Score': avg_polarity_score,
        'Tariff Mentions': tariff_mentions,
        'Tariff % of Doc': tariff_pct_of_doc,
        'Tariff Polarity Score': tariff_polarity_score
    }
    
    return summary

In [ ]:
filenames = [
    'FOMC_minutes_2018-2019',
    'FOMC_minutes_2024-2025',
    'FOMC_pressconference_2018-2019',
    'FOMC_pressconference_2024-2025',
    'FED_speech_2018-2019',
    'FED_speech_2024-2025']

for filename in filenames:
    results = []
    with open(filename +'.json', 'r', encoding='utf-8') as f:
        data = json.load(f)
    
    for doc in data:
        name = "None"
        if 'filename' in doc:
            name = doc['title']
        result = analyze_document(date = doc['date'], filename= name, full_text = doc['content'])
        results.append(result)

    final_df = pd.DataFrame(results)

    # Save the polarity result to CSV file
    final_df.to_csv("FOMC-RoBERTa_" + filename + '_analysis.csv', index=False)
    print(final_df)

In [21]:
results = []
filename = 'FED_speech_2024-2025'
with open(filename +'.json', 'r', encoding='utf-8') as f:
    data = json.load(f)

for doc in data:
    result = analyze_document(date = doc['date'], filename= "minute", full_text = doc['content'])
    results.append(result)

final_df = pd.DataFrame(results)

# Save the polarity result to CSV file
final_df.to_csv("FOMC-RoBERTa_" + filename + '_analysis.csv', index=False)
print(final_df)

    Meeting Date Filename  Total Segments  Hawkish %   Dovish %  Neutral %  \
0     2024-12-03   minute             101  57.425743  42.574257  14.802816   
1     2024-12-02   minute              66  40.909091  59.090909  29.234328   
2     2024-11-22   minute             107  71.962617  28.037383   0.030091   
3     2024-11-20   minute             122  68.032787  31.967213   8.491539   
4     2024-11-20   minute              90  41.111111  58.888889  24.790813   
..           ...      ...             ...        ...        ...        ...   
140   2025-02-04   minute              95  61.052632  38.947368  13.909165   
141   2025-01-31   minute             169  65.680473  34.319527   3.289300   
142   2025-01-09   minute             112  61.607143  38.392857   7.168364   
143   2025-01-08   minute             126  46.825397  53.174603  11.901928   
144   2025-01-06   minute             132  56.818182  43.181818  10.020210   

     Polarity Score  Tariff Mentions  Tariff % of Doc  Tariff P